## **Data cleaning and Preprocessing**

This section aims to clean, organize, and preprocess the dataset to improve data quality and prepare it for exploratory data analysis and machine learning model development.

### **1. Setup**

Import the required libraries and load the dataset for data processing, analysis, and model development.

In [46]:
#Basic setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.linear_model as lm
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#Show all columns, without hiding any
pd.set_option('display.max_columns', None)

#Set the chart background to white with a grid for easier observation and comparison of data values.
sns.set(style="whitegrid")

In [47]:
#Load the dataset
path = "../Data\\Raw\\spotify_data.csv"
df = pd.read_csv(path)

#Quick look at the data
df.head()

,track_id,track_name,track_number,track_popularity,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type,track_duration_min
0,3EJS5LyekDim1Tf5rBFmZl,Trippy Mane (ft. Project Pat),4,0,True,Diplo,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.55
1,1oQW6G2ZiwMuHqlPpP27DB,OMG!,1,0,True,Yelawolf,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,OMG!,2025-10-31,1,single,3.07
2,7mdkjzoIYlf1rx9EtBpGmU,Hard 2 Find,1,4,True,Riff Raff,48,193302,NaN,3E3zEAL8gUYWaLYB9L7gbp,Hard 2 Find,2025-10-31,1,single,2.55
3,67rW0Zl7oB3qEpD5YWWE5w,Still Get Like That (ft. Project Pat & Starrah),8,30,True,Diplo,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.69
4,15xptTfRBrjsppW0INUZjf,ride me like a harley,2,0,True,Rumelis,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,come closer / ride me like a harley,2025-10-30,2,single,2.39


In [48]:
df_cleaned = df.copy()

print("Initial shape of df_cleaned:", df_cleaned.shape)
display(df_cleaned.head())

Initial shape of df_cleaned: (8582, 15)


,track_id,track_name,track_number,track_popularity,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type,track_duration_min
0,3EJS5LyekDim1Tf5rBFmZl,Trippy Mane (ft. Project Pat),4,0,True,Diplo,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.55
1,1oQW6G2ZiwMuHqlPpP27DB,OMG!,1,0,True,Yelawolf,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,OMG!,2025-10-31,1,single,3.07
2,7mdkjzoIYlf1rx9EtBpGmU,Hard 2 Find,1,4,True,Riff Raff,48,193302,NaN,3E3zEAL8gUYWaLYB9L7gbp,Hard 2 Find,2025-10-31,1,single,2.55
3,67rW0Zl7oB3qEpD5YWWE5w,Still Get Like That (ft. Project Pat & Starrah),8,30,True,Diplo,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.69
4,15xptTfRBrjsppW0INUZjf,ride me like a harley,2,0,True,Rumelis,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,come closer / ride me like a harley,2025-10-30,2,single,2.39


### **2.Data Cleaning and Preprocessing**


*2.1. Data Cleaning*

Clean the dataset by removing duplicate, missing, invalid, and unnecessary data before analysis and modeling.

- Remove Duplicate Data

In [49]:
# 1. REMOVE DUPLICATED ROWS

duplicate_count = df_cleaned.duplicated().sum()

df_cleaned = df_cleaned.drop_duplicates()

print(f"Removed {duplicate_count} duplicated rows.")
print("Dataset shape after removing duplicates:", df_cleaned.shape)

Removed 0 duplicated rows.
Dataset shape after removing duplicates: (8582, 15)


- Handle Missing Values

In [ ]:
# 2. HANDLE MISSING CATEGORICAL VALUES

df_cleaned['artist_genres'] = df_cleaned['artist_genres'].fillna('Unknown')
df_cleaned['album_type'] = df_cleaned['album_type'].fillna('Unknown')

print("Missing values in artist_genres and album_type filled with 'Unknown'.")

print("\nMissing values after handling categorical columns:")
print(df_cleaned[['artist_genres', 'album_type']].isnull().sum())

Missing values in artist_genres and album_type filled with 'Unknown'.

Missing values after handling categorical columns:
artist_genres    0
album_type       0
dtype: int64


In [ ]:
# 3. DROP ROWS WITH MISSING IMPORTANT VALUES

required_cols = [
    'track_popularity',
    'artist_popularity',
    'artist_followers',
    'track_duration_min',
    'album_total_tracks',
    'track_number',
    'release_year',
    'release_month'
]

before_drop = df_cleaned.shape[0]

df_cleaned = df_cleaned.dropna(subset=required_cols)

after_drop = df_cleaned.shape[0]

print(f"Dropped {before_drop - after_drop} rows due to missing important values.")
print("Dataset shape after dropping missing values:", df_cleaned.shape)

Dropped 0 rows due to missing important values.
Dataset shape after dropping missing values: (8582, 17)


- Remove Invalid Data

In [ ]:
# 4. REMOVE INVALID POPULARITY VALUES

before_filter = df_cleaned.shape[0]

df_cleaned = df_cleaned[
    (df_cleaned['track_popularity'] >= 0) &
    (df_cleaned['track_popularity'] <= 100) &
    (df_cleaned['artist_popularity'] >= 0) &
    (df_cleaned['artist_popularity'] <= 100)
]

after_filter = df_cleaned.shape[0]

print(f"Removed {before_filter - after_filter} rows with invalid popularity values.")
print("Dataset shape after filtering popularity values:", df_cleaned.shape)

Removed 0 rows with invalid popularity values.
Dataset shape after filtering popularity values: (8582, 17)


In [ ]:
# 5. REMOVE INVALID DURATION AND FOLLOWERS

before_filter = df_cleaned.shape[0]

df_cleaned = df_cleaned[
    (df_cleaned['track_duration_min'] > 0) &
    (df_cleaned['artist_followers'] >= 0)
]

after_filter = df_cleaned.shape[0]

print(f"Removed {before_filter - after_filter} rows with invalid duration or followers.")
print("Dataset shape after filtering duration and followers:", df_cleaned.shape)

Removed 0 rows with invalid duration or followers.
Dataset shape after filtering duration and followers: (8582, 17)


In [ ]:
# 6. REMOVE INVALID ALBUM TRACK INFORMATION

before_filter = df_cleaned.shape[0]

df_cleaned = df_cleaned[
    (df_cleaned['album_total_tracks'] > 0) &
    (df_cleaned['track_number'] > 0) &
    (df_cleaned['track_number'] <= df_cleaned['album_total_tracks'])
]

after_filter = df_cleaned.shape[0]

print(f"Removed {before_filter - after_filter} rows with invalid album track information.")
print("Dataset shape after filtering album track information:", df_cleaned.shape)

Removed 0 rows with invalid album track information.
Dataset shape after filtering album track information: (8582, 17)


- Remove Unnecessary Features

In [ ]:
# 7. DROP UNNECESSARY COLUMNS

# Columns that are not needed for the prediction model
columns_to_drop = [
    # ID columns
    'track_id',
    'album_id',
    'artist_id',

    # Text/name columns
    'track_name',
    'album_name',
    'artist_name',

    # Raw date column
    'album_release_date',

    # Raw genre column
    'artist_genres',

    # Target-related column
    'popularity_level'
]

# Drop only columns that actually exist in the DataFrame to avoid KeyError
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]

df.drop(columns=existing_columns_to_drop, inplace=True)

print("Dropped unnecessary columns:")
print(existing_columns_to_drop)

print("\nRemaining columns:")
print(df.columns.tolist())

Dropped unnecessary columns:
['track_id', 'album_id', 'track_name', 'album_name', 'artist_name', 'album_release_date', 'artist_genres']

Remaining columns:
['track_number', 'track_popularity', 'explicit', 'artist_popularity', 'artist_followers', 'album_total_tracks', 'album_type', 'track_duration_min']


In [ ]:
# 8. Drop unecessary columns
drop_cols = [
    'is_hit',
    'track_id',
    'track_name',
    'artist_name',
    'album_name',
    'album_release_date'
]

df_cleaned = df_cleaned.drop(columns=drop_cols, errors='ignore')

*2.2. Data Formatting and Type Conversion*

Standardize data formats and convert data types to support analysis and model training.

In [ ]:
# 9. CONVERT RELEASE DATE

df_cleaned['album_release_date'] = pd.to_datetime(
    df_cleaned['album_release_date'],
    errors='coerce'
)

df_cleaned['release_year'] = df_cleaned['album_release_date'].dt.year
df_cleaned['release_month'] = df_cleaned['album_release_date'].dt.month

print("album_release_date converted to datetime.")
print("release_year and release_month created.")

display(df_cleaned[['album_release_date', 'release_year', 'release_month']].head())

album_release_date converted to datetime.
release_year and release_month created.


,album_release_date,release_year,release_month
0,2025-10-31,2025,10
1,2025-10-31,2025,10
2,2025-10-31,2025,10
3,2025-10-31,2025,10
4,2025-10-30,2025,10


In [ ]:
# 10. CONVERT NUMERICAL COLUMNS

numeric_cols = [
    'track_popularity',
    'artist_popularity',
    'artist_followers',
    'album_total_tracks',
    'track_number',
    'track_duration_min'
]

for col in numeric_cols:
    if col in df_cleaned.columns:
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

print("Numerical columns converted successfully.")

print("\nData types of numerical columns:")
print(df_cleaned[numeric_cols].dtypes)

Numerical columns converted successfully.

Data types of numerical columns:
track_popularity        int64
artist_popularity       int64
artist_followers        int64
album_total_tracks      int64
track_number            int64
track_duration_min    float64
dtype: object


*2.3. Final Data Validation*

Perform a final check after cleaning and preprocessing to ensure data consistency and quality.

In [59]:
# 11. FINAL CHECK AFTER DATA CLEANING

print("Final dataset shape after cleaning:", df_cleaned.shape)

print("\nMissing values after cleaning:")
print(df_cleaned.isnull().sum())

print("\nDuplicated rows after cleaning:")
print(df_cleaned.duplicated().sum())

print("\nData types after cleaning:")
df_cleaned.info()

display(df_cleaned.head())

Final dataset shape after cleaning: (8582, 12)

Missing values after cleaning:
track_number          0
track_popularity      0
explicit              0
artist_popularity     0
artist_followers      0
artist_genres         0
album_id              0
album_total_tracks    0
album_type            0
track_duration_min    0
release_year          0
release_month         0
dtype: int64

Duplicated rows after cleaning:


0

Data types after cleaning:
<class 'pandas.DataFrame'>
RangeIndex: 8582 entries, 0 to 8581
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   track_number        8582 non-null   int64  
 1   track_popularity    8582 non-null   int64  
 2   explicit            8582 non-null   bool   
 3   artist_popularity   8582 non-null   int64  
 4   artist_followers    8582 non-null   int64  
 5   artist_genres       8582 non-null   str    
 6   album_id            8582 non-null   str    
 7   album_total_tracks  8582 non-null   int64  
 8   album_type          8582 non-null   str    
 9   track_duration_min  8582 non-null   float64
 10  release_year        8582 non-null   int32  
 11  release_month       8582 non-null   int32  
dtypes: bool(1), float64(1), int32(2), int64(5), str(3)
memory usage: 679.0 KB


,track_number,track_popularity,explicit,artist_popularity,artist_followers,artist_genres,album_id,album_total_tracks,album_type,track_duration_min,release_year,release_month
0,4,0,True,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,9,album,1.55,2025,10
1,1,0,True,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,1,single,3.07,2025,10
2,1,4,True,48,193302,Unknown,3E3zEAL8gUYWaLYB9L7gbp,1,single,2.55,2025,10
3,8,30,True,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,9,album,1.69,2025,10
4,2,0,True,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,2,single,2.39,2025,10


### **3. Save data**

In [60]:

df_cleaned.to_csv(r'..\Data\Processed\data_cleaned.csv', index=False)